In [31]:
import pandas as pd
df= pd.read_csv('subscribers.csv')
df.head()

,user_id,signup_date,plan,age,gender,province,device,favorite_content,weekly_hours,tenure_months,churned,churn_reason
0,U1000,2024-09-11,Standard,36,Male,Buenos Aires,Computer,Series,1.5,26,No,NaN
1,U1001,2024-01-15,Premium,33,Male,Buenos Aires,Computer,Movies,14.2,34,No,NaN
2,U1002,2026-09-01,Standard,42,Male,Buenos Aires,Computer,Series,3.3,2,No,NaN
3,U1003,2025-12-05,Basic,24,Male,Other,Smart TV,Movies,4.5,11,Yes,Lack of time
4,U1004,2024-08-12,Basic,31,Male,Buenos Aires,Smart TV,Series,2.1,27,No,NaN


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   user_id           600 non-null    object 
 1   signup_date       600 non-null    object 
 2   plan              600 non-null    object 
 3   age               600 non-null    int64  
 4   gender            600 non-null    object 
 5   province          600 non-null    object 
 6   device            600 non-null    object 
 7   favorite_content  600 non-null    object 
 8   weekly_hours      600 non-null    float64
 9   tenure_months     600 non-null    int64  
 10  churned           600 non-null    object 
 11  churn_reason      265 non-null    object 
dtypes: float64(1), int64(2), object(9)
memory usage: 56.4+ KB


In [33]:
y  = df['churned'].map({'Yes' : 1, 'No': 0})

In [34]:
columnas = ['plan', 'age', 'gender', 'device', 'favorite_content', 'weekly_hours', 'tenure_months']

In [35]:
X = pd.get_dummies(df[columnas], drop_first=True)
X.head()

,age,weekly_hours,tenure_months,plan_Premium,plan_Standard,gender_Male,gender_Other,device_Mobile,device_Smart TV,device_Tablet,favorite_content_Documentaries,favorite_content_Movies,favorite_content_Reality,favorite_content_Series
0,36,1.5,26,False,True,True,False,False,False,False,False,False,False,True
1,33,14.2,34,True,False,True,False,False,False,False,False,True,False,False
2,42,3.3,2,False,True,True,False,False,False,False,False,False,False,True
3,24,4.5,11,False,False,True,False,False,True,False,False,True,False,False
4,31,2.1,27,False,False,True,False,False,True,False,False,False,False,True


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [38]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [39]:
predicciones = modelo.predict(X_test)

In [40]:
print('Accuracy:', accuracy_score(y_test, predicciones))

Accuracy: 0.6916666666666667


In [41]:
df['churn_probability'] = modelo.predict_proba(X)[:, 1] .round(2)

In [42]:
activos = df[df['churned'] == 'No']

In [43]:
activos.sort_values('churn_probability', ascending=False)[['user_id', 'plan', 'weekly_hours', 'tenure_months', 'churn_probability']].head(10)

,user_id,plan,weekly_hours,tenure_months,churn_probability
160,U1160,Basic,1.2,13,0.81
354,U1354,Basic,0.6,23,0.80
579,U1579,Basic,3.2,6,0.79
194,U1194,Basic,2.2,5,0.78
54,U1054,Basic,2.4,21,0.76
358,U1358,Basic,1.6,35,0.76
57,U1057,Basic,0.5,35,0.76
312,U1312,Standard,1.9,1,0.76
557,U1557,Basic,1.6,11,0.75
21,U1021,Basic,4.6,1,0.75


In [44]:
activos.sort_values('churn_probability', ascending=False)[['user_id', 'plan', 'weekly_hours', 'tenure_months', 'churn_probability']].to_csv('at_risk_predictions.csv', index=False)